In [1]:
"ALLOKAY"

'ALLOKAY'

In [2]:
from dotenv import load_dotenv

In [3]:
load_dotenv()

True

In [ ]:
### Chat Model Wrappers
from langchain_groq import ChatGroq
from langchain_openai import ChatOpenAI

In [ ]:
## 3. Groq Chat Model 
#model=ChatGroq(model="deepseek-r1-distill-llama-70b")
model = ChatGroq(model="openai/gpt-oss-120b", temperature=0.9)

In [ ]:
model.invoke("what is capital of USA")

AIMessage(content="<think>\nOkay, so I need to figure out what the capital of the USA is. Hmm, I'm not entirely sure, but I think it's a city I've heard of before. Let me try to remember. I know that the USA is a big country with many states, and each state has its own capital. But the country itself has a federal capital where the main government is located.\n\nI remember learning that the capital isn't one of the biggest cities like New York or Los Angeles. Maybe it's somewhere on the East Coast? I think I've heard the name Washington before, but I'm not sure if that's the state or the city. Wait, there's a state called Washington, but I also remember there's a city called Washington D.C. D.C. stands for District of Columbia, right?\n\nSo, Washington D.C. is the capital. But why is it called District of Columbia? Maybe it's a special area that isn't part of any state, created specifically to be the capital. That makes sense because I think other countries have similar setups where th

# **PARSER**

In [161]:
template=PromptTemplate(
    template="generate a prices 3 point summary from given text /n {text}",
    input_variables=["text"]
)

In [162]:
chain = template | openai_model

In [164]:
chain_parser = template | openai_model | parser

In [163]:
chain.invoke({"text":text})

AIMessage(content='1. Intelligent agents range in complexity and are found in various forms like basic control systems, human beings, firm, state or biome. They operate based on an objective function to achieve goals, taking decisions and actions over extended periods.\n2. These agents create plans to maximize their objective function\'s expected value. They are guided by reward functions in case of a reinforcement learning agent or a fitness function in the case of an evolutionary algorithm.\n3. Intelligent agents can be abstract or functional systems similar to computer programs. Real-world implementations are often referred to as software agents that perform tasks on behalf of users. An alternative term used for them is "rational agent".', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 132, 'prompt_tokens': 317, 'total_tokens': 449, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, '

In [165]:
chain_parser.invoke({"text":text})

'1. Intelligent agents in AI proactively pursue goals and perform actions, showcasing a unique form of digital agency. They vary in complexity, from simple thermostats to human beings or even larger systems like firms or states.\n   \n2. These agents operate based on an objective function encapsulating their goals. They are created to execute plans that maximize the expected value of their function upon completion. Specific agent types, like reinforcement learning agents and evolutionary algorithms, have particular behaviors guided by reward or fitness functions respectively.\n\n3. Intelligent agents are often depicted as abstract functional systems akin to computer programs. Distinctions are made between theoretical models (abstract intelligent agents) and real-world implementations, which are also related to software agents; autonomous programs that perform tasks for users. These are sometimes referred to as "rational agents."'

### **json output parser**

In [166]:
from langchain_core.output_parsers import JsonOutputParser

In [ ]:
parser=JsonOutputParser()
parser.get_format_instructions()

In [192]:
template=PromptTemplate(
    template="give me name, age and city from the provided text {text} \n {format_instructions}" ,
    input_variables=['text'],
    partial_variables={"format_instructions":parser.get_format_instructions()}
)

In [178]:
text="""hi my name is sunny savita my age is 29 and i am belong to bengaluru"""

#### **Without Chain**

In [179]:
prompt=template.format(text=text)

In [185]:
prompt

'give me name, age and city from the provided text hi my name is sunny savita my age is 29 and i am belong to bengaluru \n Return a JSON object.'

In [ ]:
# Without Chain calling llm Model directly
result=openai_model.invoke(prompt)

In [186]:
result

AIMessage(content='{\n"name": "sunny savita",\n"age": 29,\n"city": "bengaluru"\n}', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 26, 'prompt_tokens': 45, 'total_tokens': 71, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4-0613', 'system_fingerprint': None, 'finish_reason': 'stop', 'logprobs': None}, id='run-8ffe92fb-7a92-4eef-9e66-35549d7e53a8-0', usage_metadata={'input_tokens': 45, 'output_tokens': 26, 'total_tokens': 71, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [ ]:
## its a string output, we need to parse it into json format using JsonOutputParser
result.content

'{\n"name": "sunny savita",\n"age": 29,\n"city": "bengaluru"\n}'

In [ ]:
## its a string output, we need to parse it into json format using JsonOutputParser
result=parser.parse(result.content)

In [188]:
result["name"]

'sunny savita'

In [189]:
result["age"]

29

In [190]:
result["city"]

'bengaluru'

#### With CHAIN

In [ ]:
chain = template | openai_model | parser

In [ ]:
chain.invoke({"text":text})

## pass it to dataframe if u want 

{'name': 'sunny savita', 'age': 29, 'city': 'bengaluru'}

In [196]:
topic="""AI agent or simply agent), expands this concept by proactively pursuing goals, making decisions, and taking actions over extended periods, thereby exemplifying a novel form of digital agency.[1]

Intelligent agents can range from simple to highly complex. A basic thermostat or control system is considered an intelligent agent, as is a human being, or any other system that meets the same criteria—such as a firm, a state, or a biome.[2]

Intelligent agents operate based on an objective function, which encapsulates their goals. They are designed to create and execute plans that maximize the expected value of this function upon completion.[3] For example, a reinforcement learning agent has a reward function, which allows programmers to shape its desired behavior.[4] Similarly, an evolutionary algorithm's behavior is guided by a fitness function.[5]

Intelligent agents in artificial intelligence are closely related to agents in economics, and versions of the intelligent agent paradigm are studied in cognitive science, ethics, and the philosophy of practical reason, as well as in many interdisciplinary socio-cognitive modeling and computer social simulations.

Intelligent agents are often described schematically as abstract functional systems similar to computer programs. To distinguish theoretical models from real-world implementations, abstract descriptions of intelligent agents are called abstract intelligent agents. Intelligent agents are also closely related to software agents—autonomous computer programs that carry out tasks on behalf of users. They are also referred to using a term borrowed from economics: a "rational agent".[2]"""

In [197]:
template2=PromptTemplate(
    template='Give me 5 facts about {topic} \n {format_instruction}',
    input_variables=['topic'],
    partial_variables={'format_instruction': parser.get_format_instructions()}
)

In [ ]:
chain = template2 | openai_model | parser

In [199]:
chain.invoke({"topic": topic})

{'Facts': [{'Fact 1': 'Intelligent agents can range from simple to highly complex. Basic systems like a thermostat or more complex entities like a human being or a state can be considered intelligent agents.'},
  {'Fact 2': 'Intelligent agents operate based on an objective function that encapsulates their goals. They are designed to create and execute plans to maximize the expected value of this function.'},
  {'Fact 3': 'The behavior of Intelligent agents like those using reinforcement learning have a reward function designed to inform their behavior while those using an evolutionary algorithm are guided by a fitness function.'},
  {'Fact 4': 'Intelligent agents in artificial intelligence are closely related to agents in economics, and they are also studied in various fields like cognitive science, ethics, the philosophy of practical reason, and several interdisciplinary socio-cognitive modeling and computer social simulations.'},
  {'Fact 5': 'Intelligent agents are often described s

### **generate a strcuture output**

In [ ]:
## NO validation , cannot control the output with JsonOutputParser

In [200]:
from langchain.output_parsers import StructuredOutputParser, ResponseSchema

In [202]:
schema=[
    ResponseSchema(name="first_fact", description="first fact about text"),
    ResponseSchema(name="second_fact", description="second fact about text"),
    ResponseSchema(name="third_fact", description="third fact about text"),
]

In [203]:
parser=StructuredOutputParser.from_response_schemas(schema)

In [204]:
template3 = PromptTemplate(
    template='Give 3 fact about {topic} \n {format_instruction}',
    input_variables=['topic'],
    partial_variables={'format_instruction':parser.get_format_instructions()}
)

In [205]:
chain = template3 | openai_model | parser

In [206]:
result=chain.invoke({"topic":topic})

In [207]:
print(result)

{'first_fact': 'Intelligent agents can range from simple to highly complex, including systems like a basic thermostat or control system, a human being, or larger systems like a firm, a state, or a biome.', 'second_fact': 'Intelligent agents operate based on an objective function which encapsulates their goals, and they are designed to create and execute plans that maximize the expected value of this function upon completion.', 'third_fact': 'Intelligent agents in artificial intelligence are closely related to agents in economics, cognitive science, ethics, philosophy of practical reason, socio-cognitive modeling, and computer social simulations.'}


In [ ]:
print(result)

{'first_fact': 'Intelligent agents can range from simple to highly complex, including systems like a basic thermostat or control system, a human being, or larger systems like a firm, a state, or a biome.', 'second_fact': 'Intelligent agents operate based on an objective function which encapsulates their goals, and they are designed to create and execute plans that maximize the expected value of this function upon completion.', 'third_fact': 'Intelligent agents in artificial intelligence are closely related to agents in economics, cognitive science, ethics, philosophy of practical reason, socio-cognitive modeling, and computer social simulations.'}


{'first_fact': 'Intelligent agents can range from simple to highly complex, including systems like a basic thermostat or control system, a human being, or larger systems like a firm, a state, or a biome.', 'second_fact': 'Intelligent agents operate based on an objective function which encapsulates their goals, and they are designed to create and execute plans that maximize the expected value of this function upon completion.', 'third_fact': 'Intelligent agents in artificial intelligence are closely related to agents in economics, cognitive science, ethics, philosophy of practical reason, socio-cognitive modeling, and computer social simulations.'}

### **Pydantic Output parser**

In [210]:
from pydantic import BaseModel, Field
from langchain_core.output_parsers import PydanticOutputParser

In [209]:
class Person(BaseModel):
    name:str=Field(description="name of person")
    age:int=Field(gt=18,description="age of person")
    city:str=Field(description="name of the city where the person is located")

In [212]:
parser=PydanticOutputParser(pydantic_object=Person)

In [213]:
parser.get_format_instructions()

'The output should be formatted as a JSON instance that conforms to the JSON schema below.\n\nAs an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}\nthe object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.\n\nHere is the output schema:\n```\n{"properties": {"name": {"description": "name of person", "title": "Name", "type": "string"}, "age": {"description": "age of person", "exclusiveMinimum": 18, "title": "Age", "type": "integer"}, "city": {"description": "name of the city where the person is located", "title": "City", "type": "string"}}, "required": ["name", "age", "city"]}\n```'

In [ ]:
template = PromptTemplate(
    template='Generate the name, age and city of a fictional {place} person \n {format_instruction}',
    input_variables=['place'],
    partial_variables={'format_instruction':parser.get_format_instructions()}
)

In [221]:
chain = template | openai_model | parser

In [222]:
result=chain.invoke({"place":"bengaluru"})

In [224]:
result

Person(name='Rohit Sharma', age=29, city='Bengaluru')

In [223]:
print(result.name)

Rohit Sharma


In [219]:
print(result.age)

45


In [ ]:
# Vectordatabase #textsplitter
# 1. pinecone(cloud vector data base),weviate
# 2. weviate, qurdant,milvus
# 3. mongodb
# 4. faiss, chroma(in memory) #poc
# 5. on disk(persist vectordb on disk)
# 6. AWS (opensearch)

# embdding
# meta_data